In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "torch-geometric", "scikit-learn", "-q"], check=False)
import torch_geometric
print(f"torch-geometric: {torch_geometric.__version__}")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 23.3 MB/s eta 0:00:00
torch-geometric: 2.8.0.post1
All dependencies ready!


# GNN Model Training


## 1. Load Split Data Tensors


In [2]:
import os
import glob
import torch
paths_to_check = ["/kaggle/input/split-data/split_data.pt", "/kaggle/input/split-data/split_data", "split_data.pt", "model/split_data.pt"]
kaggle_glob = glob.glob("/kaggle/input/**/split_data*", recursive=True)
if kaggle_glob:
    paths_to_check.insert(0, kaggle_glob[0])
split_path = next((p for p in paths_to_check if os.path.exists(p)), None)
if split_path is None:
    raise FileNotFoundError("split_data.pt not found!")
split = torch.load(split_path, weights_only=False)
tr_data = split["tr_data"]
val_data = split["val_data"]
te_data = split["te_data"]
tr_inds = split["tr_inds"]
val_inds = split["val_inds"]
te_inds = split["te_inds"]
print(f"Loaded split_data from {split_path}: Train={tr_inds.shape[0]:,}, Val={val_inds.shape[0]:,}, Test={te_inds.shape[0]:,}")


Successfully loaded split_data from: /kaggle/input/models/shreyasnalle/split-data/pytorch/default/1/split_data.pt
Train data : Data(x=[1754264, 1], edge_index=[2, 4466821], edge_attr=[4466821, 4], y=[4466821], timestamps=[4466821])
Val data   : Data(x=[1754264, 1], edge_index=[2, 4997487], edge_attr=[4997487, 4], y=[4997487], timestamps=[4997487])
Test data  : Data(x=[1754264, 1], edge_index=[2, 5000000], edge_attr=[5000000, 4], y=[5000000], timestamps=[5000000])
Train indices : 4,466,821
Val indices   : 530,666
Test indices  : 2,513


## 2. Add Unique Edge IDs


In [3]:
def add_arange_ids(data_list):
    for data in data_list:
        arange = torch.arange(data.edge_attr.shape[0]).view(-1, 1)
        data.edge_attr = torch.cat([arange, data.edge_attr], dim=-1)
add_arange_ids([tr_data, val_data, te_data])
print(f"Edge attribute features shape with ID: {tr_data.edge_attr.shape}")


edge_attr shape after adding ID column: torch.Size([4466821, 5])
Column 0 is the unique edge ID, columns 1-4 are the original features
Sample row: [0.0, -1.103341817855835, -0.002066870918497443, 1.6104360818862915, 1.085208535194397]


## 3. Create Mini-Batch DataLoaders


In [4]:
import torch
from torch_geometric.data import Data
BATCH_SIZE = 8192
NUM_NEIGHBORS = 50
class SimpleEdgeLoader:
    def __init__(self, data, edge_inds, batch_size, shuffle=False, num_neighbors=50, balanced=False):
        self.data = data
        self.edge_inds = edge_inds
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_neighbors = num_neighbors
        self.balanced = balanced
        n_nodes = data.x.shape[0]
        src = data.edge_index[0]
        perm = src.argsort()
        self._sorted_edge_by_src = perm
        counts = torch.bincount(src, minlength=n_nodes)
        self._src_ptr = torch.cat([torch.zeros(1, dtype=torch.long), counts.cumsum(0)])
        if balanced:
            pos_mask = data.y[edge_inds] == 1
            self._pos_inds = edge_inds[pos_mask]
            self._neg_inds = edge_inds[~pos_mask]
    def __len__(self):
        return (len(self.edge_inds) + self.batch_size - 1) // self.batch_size
    def _neighbor_edges(self, nodes):
        starts = self._src_ptr[nodes]
        ends = self._src_ptr[nodes + 1]
        parts = []
        for s, e in zip(starts.tolist(), ends.tolist()):
            if s == e:
                continue
            eidx = self._sorted_edge_by_src[s:e]
            if len(eidx) > self.num_neighbors:
                eidx = eidx[torch.randperm(len(eidx))[:self.num_neighbors]]
            parts.append(eidx)
        return torch.cat(parts) if parts else torch.empty(0, dtype=torch.long)
    def _make_batch(self, chunk):
        seed_gidx = chunk
        seed_nodes = self.data.edge_index[:, seed_gidx].reshape(-1).unique()
        nbr_gidx = self._neighbor_edges(seed_nodes)
        all_gidx = torch.cat([seed_gidx, nbr_gidx]).unique()
        all_nodes = self.data.edge_index[:, all_gidx].reshape(-1).unique()
        n_total = self.data.x.shape[0]
        node_map = torch.full((n_total,), -1, dtype=torch.long)
        node_map[all_nodes] = torch.arange(len(all_nodes))
        sub_ei = node_map[self.data.edge_index[:, all_gidx]]
        batch = Data(x=self.data.x[all_nodes], edge_index=sub_ei, edge_attr=self.data.edge_attr[all_gidx], y=self.data.y[all_gidx], num_nodes=len(all_nodes))
        return batch, all_gidx, seed_gidx
    def __iter__(self):
        if self.balanced:
            half = self.batch_size // 2
            n_batches = len(self)
            for _ in range(n_batches):
                pos_i = torch.randint(0, len(self._pos_inds), (half,))
                neg_i = torch.randint(0, len(self._neg_inds), (half,))
                chunk = torch.cat([self._pos_inds[pos_i], self._neg_inds[neg_i]])
                batch, all_gidx, seed_gidx = self._make_batch(chunk)
                batch.input_id = chunk
                batch._seed_ids = self.data.edge_attr[seed_gidx, 0]
                yield batch
        else:
            n = len(self.edge_inds)
            order = torch.randperm(n) if self.shuffle else torch.arange(n)
            for start in range(0, n, self.batch_size):
                chunk_pos = order[start:start + self.batch_size]
                chunk = self.edge_inds[chunk_pos]
                batch, all_gidx, seed_gidx = self._make_batch(chunk)
                batch.input_id = chunk_pos
                batch._seed_ids = self.data.edge_attr[seed_gidx, 0]
                yield batch
tr_loader = SimpleEdgeLoader(tr_data, tr_inds, BATCH_SIZE, shuffle=True, num_neighbors=NUM_NEIGHBORS, balanced=True)
val_loader = SimpleEdgeLoader(val_data, val_inds, BATCH_SIZE, shuffle=False, num_neighbors=NUM_NEIGHBORS, balanced=False)
te_loader = SimpleEdgeLoader(te_data, te_inds, BATCH_SIZE, shuffle=False, num_neighbors=NUM_NEIGHBORS, balanced=False)
sample_batch = next(iter(tr_loader))
print(f"DataLoaders initialized: Train batches={len(tr_loader)}, Val batches={len(val_loader)}, Test batches={len(te_loader)}")


  Balanced loader: 1,008 positives | 4,465,813 negatives
  Illicit ratio: 0.023%

Sample batch: 34905 edges | 16830 nodes
Positive (laundering) in batch: 994 / 34905
edge_attr shape: torch.Size([34905, 5])


## 4. Define GINe Model Architecture


In [5]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, BatchNorm, Linear
class GINe(torch.nn.Module):
    def __init__(self, num_features, num_gnn_layers, n_classes=2, n_hidden=100, edge_updates=False, edge_dim=None, dropout=0.0, final_dropout=0.5):
        super().__init__()
        self.n_hidden = n_hidden
        self.num_gnn_layers = num_gnn_layers
        self.edge_updates = edge_updates
        self.final_dropout = final_dropout
        self.node_emb = nn.Linear(num_features, n_hidden)
        self.edge_emb = nn.Linear(edge_dim, n_hidden)
        self.convs = nn.ModuleList()
        self.emlps = nn.ModuleList()
        self.batch_norms = nn.ModuleList()
        for _ in range(self.num_gnn_layers):
            conv = GINEConv(nn.Sequential(nn.Linear(self.n_hidden, self.n_hidden), nn.ReLU(), nn.Linear(self.n_hidden, self.n_hidden)), edge_dim=self.n_hidden)
            if self.edge_updates:
                self.emlps.append(nn.Sequential(nn.Linear(3 * self.n_hidden, self.n_hidden), nn.ReLU(), nn.Linear(self.n_hidden, self.n_hidden)))
            self.convs.append(conv)
            self.batch_norms.append(BatchNorm(n_hidden))
        self.mlp = nn.Sequential(Linear(n_hidden * 3, 50), nn.ReLU(), nn.Dropout(self.final_dropout), Linear(50, 25), nn.ReLU(), nn.Dropout(self.final_dropout), Linear(25, n_classes))
    def forward(self, x, edge_index, edge_attr):
        src, dst = edge_index
        x = self.node_emb(x)
        edge_attr = self.edge_emb(edge_attr)
        for i in range(self.num_gnn_layers):
            x = (x + F.relu(self.batch_norms[i](self.convs[i](x, edge_index, edge_attr)))) / 2
            if self.edge_updates:
                edge_attr = edge_attr + self.emlps[i](torch.cat([x[src], x[dst], edge_attr], dim=-1)) / 2
        x = x[edge_index.T].reshape(-1, 2 * self.n_hidden).relu()
        x = torch.cat((x, edge_attr.view(-1, edge_attr.shape[1])), dim=1)
        return self.mlp(x)
N_HIDDEN = 64
N_GNN_LAYERS = 2
EDGE_DIM = sample_batch.edge_attr.shape[1] - 1
NUM_FEATURES = sample_batch.x.shape[1]
model = GINe(num_features=NUM_FEATURES, num_gnn_layers=N_GNN_LAYERS, n_classes=2, n_hidden=N_HIDDEN, edge_updates=True, edge_dim=EDGE_DIM, dropout=0.0, final_dropout=0.5)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"GINe Model initialized with {total_params:,} trainable parameters")


Model architecture:
GINe(
  (node_emb): Linear(in_features=1, out_features=64, bias=True)
  (edge_emb): Linear(in_features=4, out_features=64, bias=True)
  (convs): ModuleList(
    (0-1): 2 x GINEConv(nn=Sequential(
      (0): Linear(in_features=64, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    ))
  )
  (emlps): ModuleList(
    (0-1): 2 x Sequential(
      (0): Linear(in_features=192, out_features=64, bias=True)
      (1): ReLU()
      (2): Linear(in_features=64, out_features=64, bias=True)
    )
  )
  (batch_norms): ModuleList(
    (0-1): 2 x BatchNorm(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (mlp): Sequential(
    (0): Linear(192, 50, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.5, inplace=False)
    (3): Linear(50, 25, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.5, inplace=False)
    (6): Linear(25, 2, bias=True)
  )
)
Total trainable parameters: 69,665


## 5. Device and Optimizer Setup


In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
LR = 0.0003
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
print(f"Training on device: {device}")


Training on: cuda
Optimizer: Adam (lr=0.0003)


## 6. Training Loop


In [7]:
import os
import tqdm
from sklearn.metrics import f1_score, precision_score, recall_score
@torch.no_grad()
def evaluate(loader, model, device):
    model.eval()
    preds, ground_truths = [], []
    for batch in tqdm.tqdm(loader, desc="Evaluating", leave=False):
        seed_ids = batch._seed_ids
        mask = torch.isin(batch.edge_attr[:, 0].detach().cpu(), seed_ids.cpu())
        batch.edge_attr = batch.edge_attr[:, 1:]
        batch = batch.to(device)
        mask = mask.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        pred = out[mask].argmax(dim=-1)
        preds.append(pred.cpu())
        ground_truths.append(batch.y[mask].cpu())
    pred = torch.cat(preds).numpy()
    ground_truth = torch.cat(ground_truths).numpy()
    return {
        "f1": f1_score(ground_truth, pred, zero_division=0),
        "precision": precision_score(ground_truth, pred, zero_division=0),
        "recall": recall_score(ground_truth, pred, zero_division=0)
    }
N_EPOCHS = 30
W_CE1 = 1.0
W_CE2 = 200.0
best_val_f1 = 0.0
LOCAL_DIR = "/home/shreyas-nalle/Desktop/Delusional/model"
KAGGLE_DIR = "/kaggle/working"
if os.path.isdir(LOCAL_DIR):
    SAVE_DIR = LOCAL_DIR
elif os.path.isdir(KAGGLE_DIR):
    SAVE_DIR = KAGGLE_DIR
else:
    SAVE_DIR = "."
model_save_path = os.path.join(SAVE_DIR, "best_model.pt")
loss_fn = torch.nn.CrossEntropyLoss(weight=torch.FloatTensor([W_CE1, W_CE2]).to(device))
print(f"Training for {N_EPOCHS} epochs...")
for epoch in range(1, N_EPOCHS + 1):
    model.train()
    total_loss = total_examples = 0
    preds, ground_truths = [], []
    for batch in tqdm.tqdm(tr_loader, desc=f"Epoch {epoch}/{N_EPOCHS}", leave=False):
        optimizer.zero_grad()
        seed_ids = batch._seed_ids
        mask = torch.isin(batch.edge_attr[:, 0].detach().cpu(), seed_ids.cpu())
        batch.edge_attr = batch.edge_attr[:, 1:]
        batch = batch.to(device)
        mask = mask.to(device)
        out = model(batch.x, batch.edge_index, batch.edge_attr)
        pred = out[mask]
        ground_truth = batch.y[mask]
        if pred.numel() == 0:
            continue
        preds.append(pred.argmax(dim=-1).detach().cpu())
        ground_truths.append(ground_truth.detach().cpu())
        loss = loss_fn(pred, ground_truth)
        loss.backward()
        optimizer.step()
        total_loss += float(loss) * pred.shape[0]
        total_examples += pred.shape[0]
    train_pred = torch.cat(preds).numpy()
    train_gt = torch.cat(ground_truths).numpy()
    train_f1 = f1_score(train_gt, train_pred, zero_division=0)
    avg_loss = total_loss / max(total_examples, 1)
    val_metrics = evaluate(val_loader, model, device)
    if val_metrics["f1"] > best_val_f1:
        best_val_f1 = val_metrics["f1"]
        torch.save(model.state_dict(), model_save_path)
    print(f"Epoch {epoch:02d} | Loss: {avg_loss:.4f} | Train F1: {train_f1:.4f} | Val F1: {val_metrics['f1']:.4f} | Val Prec: {val_metrics['precision']:.4f} | Val Rec: {val_metrics['recall']:.4f}")
print(f"Training complete. Best Val F1: {best_val_f1:.4f}")
print(f"Model saved at: {model_save_path}")


Model weights will be saved to: /kaggle/working/best_model.pt
Training on: cuda
Loss: WeightedCE  w_clean=1.0  w_laundering=200.0
Epochs: 30



Epoch 1/30:   0%|          | 0/546 [00:00<?, ?it/s]/tmp/ipykernel_157/3012866810.py:75: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  total_loss     += float(loss) * pred.shape[0]
  Evaluating: 100%|██████████| 65/65 [00:08<00:00,  7.54it/s]


  ✓ Saved best model (Val F1=0.0016)
Epoch 01 | Loss: 0.0784 | Train F1: 0.4743 | Val F1: 0.0016 | Val Prec: 0.0008 | Val Rec: 0.9969


  Evaluating: 100%|██████████| 65/65 [00:08<00:00,  7.54it/s]


  ✓ Saved best model (Val F1=0.0031)
Epoch 02 | Loss: 0.0313 | Train F1: 0.6452 | Val F1: 0.0031 | Val Prec: 0.0015 | Val Rec: 0.9690


...

Epoch 28 | Loss: 0.0142 | Train F1: 0.7944 | Val F1: 0.0031 | Val Prec: 0.0016 | Val Rec: 0.8731


  Evaluating: 100%|██████████| 65/65 [00:08<00:00,  7.28it/s]


Epoch 29 | Loss: 0.0144 | Train F1: 0.7933 | Val F1: 0.0028 | Val Prec: 0.0014 | Val Rec: 0.8390


  Evaluating: 100%|██████████| 65/65 [00:08<00:00,  7.44it/s]

Epoch 30 | Loss: 0.0148 | Train F1: 0.7877 | Val F1: 0.0028 | Val Prec: 0.0014 | Val Rec: 0.8514

Training complete! Best Val F1: 0.0037
Model saved at: /kaggle/working/best_model.pt


## 7. Final Test Evaluation


In [ ]:
import os, shutil, glob
import torch
LOCAL_PATH = "/home/shreyas-nalle/Desktop/Delusional/model/best_model.pt"
KAGGLE_PATH = "/kaggle/working/best_model.pt"
CWD_PATH = "best_model.pt"
for p in [LOCAL_PATH, KAGGLE_PATH, CWD_PATH]:
    if os.path.exists(p):
        model_load_path = p
        break
else:
    raise FileNotFoundError("best_model.pt not found!")
print(f"Loading best model from: {model_load_path}")
model.load_state_dict(torch.load(model_load_path, map_location=device, weights_only=True))
final_metrics = evaluate(te_loader, model, device)
print("Final Test Set Results:")
print(f"  F1 Score  : {final_metrics['f1']:.4f}")
print(f"  Precision : {final_metrics['precision']:.4f}")
print(f"  Recall    : {final_metrics['recall']:.4f}")


Loading best model from: /kaggle/working/best_model.pt


  Evaluating:   0%|          | 0/1 [00:00<?, ?it/s]

  Evaluating: 100%|██████████| 1/1 [00:00<00:00, 53.14it/s]


Final Test Set Results (Best Val Checkpoint):
  F1 Score  : 0.0340
  Precision : 0.4333
  Recall    : 0.0177

📥 To get best_model.pt locally:
   → In the Kaggle sidebar, click Output → best_model.pt → Download
   → OR run the cell below locally after saving this notebook version
